In [1]:
pip install imbalanced-learn


Note: you may need to restart the kernel to use updated packages.


In [1]:
import pandas as pd
import re
import string
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy.sparse import hstack
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
import joblib


In [3]:
# 1. Load & clean data
# -------------------------------
df = pd.read_csv(r"C:\Users\USER\OneDrive\Desktop\fake_job\fake_job_postings.csv")
df.drop(columns=['salary_range', 'department'], inplace=True)

for col in ['company_profile', 'requirements', 'benefits', 'employment_type',
            'required_experience', 'required_education', 'industry', 'function']:
    df[col] = df[col].fillna("Not Provided")

df = df.dropna(subset=['location', 'description'])


In [5]:
# -----------------------------
# 2. Offline stopwords list
# -----------------------------
stop_words = set("""
a about above after again against all am an and any are as at be because been before
being below between both but by could did do does doing down during each few for from
further had has have having he her here hers herself him himself his how i if in into
is it its itself just me more most my myself no nor not of off on once only or other
our ours ourselves out over own same she should so some such than that the their theirs
them themselves then there these they this those through to too under until up very was
we were what when where which while who whom why will with you your yours yourself
yourselves
""".split())

In [7]:
# -----------------------------
# 3. Text cleaning function
# -----------------------------
def clean_text(text):
    text = str(text).lower()
    text = re.sub(f"[{re.escape(string.punctuation)}]", "", text)
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    text = ' '.join([word for word in text.split() if word not in stop_words])
    return text

# -----------------------------
# 4. Combine text columns
# -----------------------------
df['text'] = df[['title', 'company_profile', 'description', 'requirements', 'benefits']].astype(str).agg(' '.join, axis=1)
df['text'] = df['text'].apply(clean_text)

In [9]:
#5. TF-IDF vectorization
# -----------------------------
vectorizer = TfidfVectorizer(max_features=5000)
X_text = vectorizer.fit_transform(df['text'])


In [11]:
# -----------------------------
# 6. New textual features
# -----------------------------
# Text length
df['text_length'] = df['text'].apply(lambda x: len(x.split()))
# Number of words in description
df['num_words'] = df['description'].apply(lambda x: len(str(x).split()))
# Number of capitalized words
df['num_caps'] = df['description'].apply(lambda x: sum(1 for w in str(x).split() if w.isupper()))
# Suspicious phrases
suspicious_phrases = ["urgent hire", "quick money", "work from home", "make money fast", "no experience required"]
df['suspicious_phrase'] = df['description'].apply(lambda x: int(any(p in str(x).lower() for p in suspicious_phrases)))

# -----------------------------
# 7. Structured features + textual features
# -----------------------------
X_structured = df[['telecommuting', 'has_company_logo', 'has_questions',
                   'text_length', 'num_words', 'num_caps', 'suspicious_phrase']]
X = hstack([X_text, X_structured])
y = df['fraudulent']


In [13]:
# -----------------------------
# 8. Train-test split
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# -----------------------------
# -----------------------------
# 9. Apply SMOTE
# -----------------------------
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)


In [19]:
# 10. Train models
# -----------------------------
# Logistic Regression
log_model = LogisticRegression(max_iter=5000, class_weight='balanced')
log_model.fit(X_train_res.toarray(), y_train_res)  

LogisticRegression(class_weight='balanced', max_iter=5000)

In [21]:
# XGBoost
xgb_model = XGBClassifier(eval_metric='logloss')
xgb_model.fit(X_train_res, y_train_res)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [25]:
# -----------------------------
# 11. Evaluate
# -----------------------------
proba_lr = log_model.predict_proba(X_test.toarray())[:, 1]
proba_xgb = xgb_model.predict_proba(X_test)[:, 1]
proba_avg = (proba_lr + proba_xgb) / 2
y_pred = (proba_avg >= 0.45).astype(int)


In [27]:
from sklearn.metrics import confusion_matrix, classification_report

# -----------------------------
# Logistic Regression
# -----------------------------
y_pred_log = log_model.predict(X_test.toarray())  # LogisticRegression needs dense input
print("Logistic Regression Confusion Matrix:\n", confusion_matrix(y_test, y_pred_log))
print("Logistic Regression Classification Report:\n", classification_report(y_test, y_pred_log))

# -----------------------------
# XGBoost
# -----------------------------
y_pred_xgb = xgb_model.predict(X_test)  # XGBoost works with sparse or dense
print("XGBoost Confusion Matrix:\n", confusion_matrix(y_test, y_pred_xgb))
print("XGBoost Classification Report:\n", classification_report(y_test, y_pred_xgb))


Logistic Regression Confusion Matrix:
 [[3224  114]
 [  20  149]]
Logistic Regression Classification Report:
               precision    recall  f1-score   support

           0       0.99      0.97      0.98      3338
           1       0.57      0.88      0.69       169

    accuracy                           0.96      3507
   macro avg       0.78      0.92      0.83      3507
weighted avg       0.97      0.96      0.97      3507

XGBoost Confusion Matrix:
 [[3329    9]
 [  46  123]]
XGBoost Classification Report:
               precision    recall  f1-score   support

           0       0.99      1.00      0.99      3338
           1       0.93      0.73      0.82       169

    accuracy                           0.98      3507
   macro avg       0.96      0.86      0.90      3507
weighted avg       0.98      0.98      0.98      3507



In [29]:
# Save XGBoost in JSON
xgb_model.get_booster().save_model("xgb_model.json")
print("✅ XGBoost model saved as JSON.")



✅ XGBoost model saved as JSON.


In [31]:
# -----------------------------
# 11. Ensemble model
# -----------------------------
ensemble_model = VotingClassifier(
    estimators=[('lr', log_model), ('xgb', xgb_model)],
    voting='soft'
)
ensemble_model.fit(X_train_res.toarray(), y_train_res)


VotingClassifier(estimators=[('lr',
                              LogisticRegression(class_weight='balanced',
                                                 max_iter=5000)),
                             ('xgb',
                              XGBClassifier(base_score=None, booster=None,
                                            callbacks=None,
                                            colsample_bylevel=None,
                                            colsample_bynode=None,
                                            colsample_bytree=None, device=None,
                                            early_stopping_rounds=None,
                                            enable_categorical=False,
                                            eval_metric='logloss',
                                            feature_types=None, gamma=None,
                                            grow_pol...
                                            importance_type=None,
                                            interaction_constraints=None,
                                            learning_rate=None, max_bin=None,
                                            max_cat_threshold=None,
                                            max_cat_to_onehot=None,
                                            max_delta_step=None, max_depth=None,
                                            max_leaves=None,
                                            min_child_weight=None, missing=nan,
                                            monotone_constraints=None,
                                            multi_strategy=None,
                                            n_estimators=None, n_jobs=None,
                                            num_parallel_tree=None,
                                            random_state=None, ...))],
                 voting='soft')

In [33]:
# -----------------------------
# 12. Calibrate ensemble
# -----------------------------
calibrated_model = CalibratedClassifierCV(
    estimator=ensemble_model,
    method='isotonic',
    cv=5
)
calibrated_model.fit(X_train_res.toarray(), y_train_res)


CalibratedClassifierCV(cv=5,
                       estimator=VotingClassifier(estimators=[('lr',
                                                               LogisticRegression(class_weight='balanced',
                                                                                  max_iter=5000)),
                                                              ('xgb',
                                                               XGBClassifier(base_score=None,
                                                                             booster=None,
                                                                             callbacks=None,
                                                                             colsample_bylevel=None,
                                                                             colsample_bynode=None,
                                                                             colsample_bytree=None,
                                                                             device=None,
                                                                             early_stopping_rounds=None,
                                                                             enable_categorical=False,
                                                                             eval_metric='logloss',...
                                                                             importance_type=None,
                                                                             interaction_constraints=None,
                                                                             learning_rate=None,
                                                                             max_bin=None,
                                                                             max_cat_threshold=None,
                                                                             max_cat_to_onehot=None,
                                                                             max_delta_step=None,
                                                                             max_depth=None,
                                                                             max_leaves=None,
                                                                             min_child_weight=None,
                                                                             missing=nan,
                                                                             monotone_constraints=None,
                                                                             multi_strategy=None,
                                                                             n_estimators=None,
                                                                             n_jobs=None,
                                                                             num_parallel_tree=None,
                                                                             random_state=None, ...))],
                                                  voting='soft'),
                       method='isotonic')

In [35]:
# -----------------------------
# 13. Evaluate ensemble
# -----------------------------
y_pred_ensemble = calibrated_model.predict(X_test.toarray())
print("Ensemble Confusion Matrix:\n", confusion_matrix(y_test, y_pred_ensemble))
print("Ensemble Classification Report:\n", classification_report(y_test, y_pred_ensemble))


Ensemble Confusion Matrix:
 [[3332    6]
 [  51  118]]
Ensemble Classification Report:
               precision    recall  f1-score   support

           0       0.98      1.00      0.99      3338
           1       0.95      0.70      0.81       169

    accuracy                           0.98      3507
   macro avg       0.97      0.85      0.90      3507
weighted avg       0.98      0.98      0.98      3507



In [37]:
# -----------------------------
# 14. Save calibrated ensemble & vectorizer
# -----------------------------
joblib.dump(calibrated_model, "best_model_ensemble.pkl")
joblib.dump(vectorizer, "vectorizer.pkl")
print("✅ Ensemble model and vectorizer saved successfully.")

✅ Ensemble model and vectorizer saved successfully.
